# Memories con LangChain — Demo del formador (02)

**Píldora:** Orquestación de flujos con LangChain/LlamaIndex: Memories · 60 minutos.

### Qué demuestra esta sesión
1. Sin contexto guardado, el LLM **no recuerda** nada del turno anterior (las llamadas son *stateless*).
2. Con un `checkpointer` de LangGraph y un `thread_id` por conversación, el historial **se guarda y se recupera solo**.
3. Dos `thread_id` distintos = dos sesiones **aisladas**.
4. El historial se puede **inspeccionar** y **borrar**.

### Coste de la demo
Son **4 llamadas cortas** al modelo con un prompt de sistema que limita la respuesta a unas 60 palabras: coste residual (Groq tiene nivel gratuito; con `gpt-4o-mini` es una fracción de céntimo). Recuerda que **cada turno reenvía todo el historial**: en conversaciones largas el coste crece.

> **Dos modos:** si detecta una API key → **modo con API** (LLM real). Si no → **modo sin API** (modelo determinista: la memoria sigue siendo la de LangChain, solo se simula la generación del texto).

### Recorridos de la píldora

| | Recorrido **A** (recomendado) | Recorrido **B** (opcional) |
|---|---|---|
| Qué es | Clase sencilla, sin claves ni cuentas | Demo técnica con LLM real |
| Cómo | Ejecuta este notebook **sin clave** (modo determinista) + reto `03` | Ejecuta este notebook **con** `GROQ_API_KEY` |
| Necesitas | Solo las celdas del notebook | Cuenta gratuita de [Groq](https://console.groq.com/keys) |

Hay además una **app Streamlit** interactiva con la misma demo: `streamlit run app.py` (ver README).

📖 [Memoria (conceptos)](https://docs.langchain.com/oss/python/concepts/memory) · [Memoria a corto plazo](https://docs.langchain.com/oss/python/langchain/short-term-memory) · [RunnableWithMessageHistory](https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory) · [Memory en LlamaIndex](https://developers.llamaindex.ai/python/examples/memory/memory/)

## Paso 1 · Instalación (versiones fijadas)

Primera celda obligatoria. En Colab, si pide reiniciar el entorno: **Runtime → Restart session** y vuelve a *Run all*.

In [1]:
%pip install -q "langchain==1.4.3" "langchain-core==1.6.6" "langgraph==1.2.12" "langchain-openai==1.6.7" "langchain-groq==1.1.3"

Note: you may need to restart the kernel to use updated packages.


## Paso 2 · Clave de API y modo de ejecución

La clave **nunca se escribe en el notebook**. Se busca en este orden:

1. Variable de entorno (`GROQ_API_KEY` / `OPENAI_API_KEY`).
2. **Secretos de Colab** (icono de candado → *Add secret*).
3. Último recurso en Colab: pregunta segura por pantalla (`getpass`).

Si no hay clave, la demo continúa en **modo sin API**.

In [2]:
import os

def _en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _buscar_clave(nombre):
    """Variable de entorno o secreto de Colab (sin preguntar por pantalla)."""
    valor = os.environ.get(nombre)
    if valor:
        return valor
    if _en_colab():
        try:
            from google.colab import userdata
            return userdata.get(nombre)          # lanza si el secreto no existe
        except Exception:
            return None
    return None

def _preguntar_clave(nombre):
    """Solo en Colab: pide la clave de forma segura. Enter => seguir sin API."""
    if not _en_colab():
        return None
    import getpass
    try:
        return getpass.getpass(nombre + " (Enter para continuar sin API): ").strip() or None
    except Exception:
        return None

proveedor_solicitado = os.environ.get("PROVEEDOR")   # opcional: "groq" u "openai"

if proveedor_solicitado == "groq":
    proveedor, clave = "groq", _buscar_clave("GROQ_API_KEY") or _preguntar_clave("GROQ_API_KEY")
elif proveedor_solicitado == "openai":
    proveedor, clave = "openai", _buscar_clave("OPENAI_API_KEY") or _preguntar_clave("OPENAI_API_KEY")
else:
    clave = _buscar_clave("GROQ_API_KEY")
    proveedor = "groq" if clave else None
    if not clave:
        clave = _buscar_clave("OPENAI_API_KEY")
        proveedor = "openai" if clave else None
    if not clave:                                # último recurso: una sola pregunta
        clave = _preguntar_clave("GROQ_API_KEY")
        proveedor = "groq" if clave else None

if clave:
    os.environ["GROQ_API_KEY" if proveedor == "groq" else "OPENAI_API_KEY"] = clave
    MODO = "con_api"
    print("Modo: CON API  ·  proveedor:", proveedor, " ·  clave oculta (" + str(len(clave)) + " caracteres)")
else:
    MODO = "sin_api"
    print("Modo: SIN API (demo determinista)")
    print("  -> Para el LLM real: añade GROQ_API_KEY o OPENAI_API_KEY en Secrets de Colab y reinicia el runtime.")

Modo: CON API  ·  proveedor: groq  ·  clave oculta (56 caracteres)


## Paso 3 · Configuración de la memoria

La memoria de la píldora tiene dos piezas:

| Pieza | Papel | Ejemplo |
|-------|-------|---------|
| `checkpointer` | **El almacén**: persiste el estado de cada conversación | `InMemorySaver()` (en RAM) |
| `thread_id` | **La etiqueta de la sesión**: separa una conversación de otra | `"sesion-alex"`, `"sesion-nueva"` |

```
   Turno n
 ┌───────────────┐
 │    Usuario    │── mensaje ─▶ ┌──────────────────────────────────────────┐
 └───────────────┘              │  HISTORIAL DEL HILO  (thread_id)         │
                                │  [(user, "Me llamo Alex..."), (ai, ...)] │
      prompt = system           │        ▲                    │            │
      + historial               │        │ respuesta          │ se recupera│
      + mensaje nuevo           │        │ guardada           │ todo       │
 ┌───────────────┐              │        │                    ▼            │
 │      LLM      │◀── resp. ──  │   [ preparar prompt ] ──▶ [ LLM ]        │
 └───────────────┘              └──────────────────────────────────────────┘
                                 checkpointer = InMemorySaver (el almacén)

   Sesión nueva = otro thread_id = historial vacío = sin contexto previo
```

In [3]:
from langgraph.checkpoint.memory import InMemorySaver

# El almacén de memoria: guarda un checkpoint por cada hilo (thread_id).
checkpointer = InMemorySaver()

def hilo(id_sesion):
    """Configuración que identifica la conversación ante LangGraph."""
    return {"configurable": {"thread_id": id_sesion}}

print("Checkpointer listo:", type(checkpointer).__name__)

Checkpointer listo: InMemorySaver


## Paso 4 · Crear el agente con memoria

`create_agent(..., checkpointer=...)` es el enfoque **vigente** de LangChain 1.x para memoria a corto plazo (antes se usaban `ConversationChain` / `ConversationBufferMemory`, hoy **obsoletos**).

- **Modo con API:** modelo real (`groq:openai/gpt-oss-120b` o `openai:gpt-4o-mini`).
- **Modo sin API:** `ModeloDeterminista`, una clase propia de LangChain (`BaseChatModel`) que responde en función del historial que recibe. **La memoria no la implementa el modelo:** la gestionan LangChain/LangGraph en los dos modos.

In [4]:
from typing import Any, Optional
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult

class ModeloDeterminista(BaseChatModel):
    """Modelo simulado SIN API: responde según el historial que recibe.

    Solo sustituye la generación del texto. El guardado, la recuperación,
    el aislamiento y el borrado los sigue haciendo LangChain/LangGraph.
    """

    @property
    def _llm_type(self):
        return "modelo-determinista"

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        historial = "\n".join(str(m.content) for m in messages)
        ultimo_usuario = ""
        for m in reversed(messages):
            if m.type == "human":
                ultimo_usuario = str(m.content)
                break

        con_contexto = ("Alex" in historial) and ("Python" in historial)

        if "CSV" in ultimo_usuario.upper():
            if con_contexto:
                respuesta = ("Hola Alex, como trabajas con Python y prefieres las explicaciones "
                             "sencillas: pandas.read_csv('fichero.csv') y listo.")
            else:
                respuesta = "Puedes usar pandas.read_csv('fichero.csv')."
        elif "Me llamo" in ultimo_usuario or "Alex" in ultimo_usuario:
            respuesta = "Encantado, Alex. Anotado: Python y explicaciones sencillas."
        else:
            respuesta = "No tengo contexto previo de esta conversacion."

        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=respuesta))])

print("ModeloDeterminista definido:", ModeloDeterminista()._llm_type)

ModeloDeterminista definido: modelo-determinista


In [5]:
from langchain.agents import create_agent

if MODO == "con_api":
    modelo = "groq:openai/gpt-oss-120b" if proveedor == "groq" else "openai:gpt-4o-mini"
else:
    modelo = ModeloDeterminista()

agente = create_agent(
    model=modelo,
    tools=[],                     # solo conversación: sin herramientas
    system_prompt=("Eres el asistente de una píldora formativa. Responde SIEMPRE en español, "
                   "en menos de 60 palabras y con explicaciones sencillas."),
    checkpointer=checkpointer,    # <-- aquí empieza la memoria
)

print("Agente creado. Modelo:", modelo if isinstance(modelo, str) else "ModeloDeterminista (sin API)")

Agente creado. Modelo: groq:openai/gpt-oss-120b


## Paso 5 · Primer turno (hilo `sesion-alex`)

> «Me llamo Alex, trabajo con Python y prefiero explicaciones sencillas».

> Con API → LLM real · Sin API → modelo determinista. **La memoria es idéntica en ambos recorridos.**

In [6]:
hilo_alex = hilo("sesion-alex")

turno1 = agente.invoke(
    {"messages": [{"role": "user",
                   "content": "Me llamo Alex, trabajo con Python y prefiero explicaciones sencillas."}]},
    hilo_alex,                    # <-- la config identifica la conversación
)

print("ALEX  :", "Me llamo Alex, trabajo con Python y prefiero explicaciones sencillas.")
print("ASIST.:", turno1["messages"][-1].content)

ALEX  : Me llamo Alex, trabajo con Python y prefiero explicaciones sencillas.
ASIST.: ¡Hola Alex! Encantado de conocerte. ¿En qué tema de Python necesitas ayuda? Puedo explicarte conceptos, resolver dudas o guiarte paso a paso. ¡Dime qué te gustaría aprender!


## Paso 6 · Segundo turno **en el mismo hilo**

Misma pregunta de antes («¿Cómo puedo leer un archivo CSV?»), pero ahora el agente **debería** aprovechar el contexto: saber que te llamas, que usas Python y que quieres explicaciones sencillas.

> Con API → LLM real · Sin API → modelo determinista. **La memoria es idéntica en ambos recorridos.**

In [7]:
turno2 = agente.invoke(
    {"messages": [{"role": "user", "content": "¿Cómo puedo leer un archivo CSV?"}]},
    hilo_alex,                    # <-- MISMO thread_id: el historial está disponible
)

print("ALEX  :", "¿Cómo puedo leer un archivo CSV?")
print("ASIST.:", turno2["messages"][-1].content)

ALEX  : ¿Cómo puedo leer un archivo CSV?
ASIST.: Usa el módulo `csv` o la librería `pandas`.

**Con `csv`:**  
```python
import csv
with open('archivo.csv', newline='') as f:
    lector = csv.reader(f)
    for fila in lector:
        print(fila)
```

**Con `pandas`:**  
```python
import pandas as pd
df = pd.read_csv('archivo.csv')
print(df.head())
```


## Paso 7 · Segunda sesión independiente

Otro usuario, otra conversación: **otro `thread_id`**. Misma pregunta, pero aquí **no debe** aparecer el contexto de Alex.

> Con API → LLM real · Sin API → modelo determinista. **La memoria es idéntica en ambos recorridos.**

In [8]:
hilo_nuevo = hilo("sesion-nueva")     # <-- otro thread_id = otra sesión

turno3 = agente.invoke(
    {"messages": [{"role": "user", "content": "¿Cómo puedo leer un archivo CSV?"}]},
    hilo_nuevo,
)

print("NUEVO :", "¿Cómo puedo leer un archivo CSV?")
print("ASIST.:", turno3["messages"][-1].content)

NUEVO : ¿Cómo puedo leer un archivo CSV?
ASIST.: Puedes usar Python:

```python
import pandas as pd
df = pd.read_csv('ruta/archivo.csv')
print(df.head())
```

`read_csv` abre el archivo, lo convierte en una tabla (DataFrame) y `head()` muestra las primeras filas. También puedes usar el módulo `csv` para leer fila a fila.


## Paso 8 · Inspeccionar el historial / estado

`agent.get_state(config)` devuelve el **estado persistido del hilo**: ahí vive la memoria. No confundas:

- `turno2["messages"][-1].content` → solo la **última respuesta**.
- `get_state(...).values["messages"]` → **el historial completo** (usuario + asistente, en orden).

In [9]:
def mostrar_historial(id_sesion):
    estado = agente.get_state(hilo(id_sesion))
    mensajes = estado.values.get("messages", [])
    print("\n=== Historial de '" + id_sesion + "': " + str(len(mensajes)) + " mensajes ===")
    for m in mensajes:
        print("  [" + str(m.type) + "] " + str(m.content))
    return mensajes

mensajes_alex = mostrar_historial("sesion-alex")
mensajes_nuevo = mostrar_historial("sesion-nueva")

assert any("Me llamo Alex" in str(m.content) for m in mensajes_alex), \
    "la sesión Alex debe conservar su primer turno"
assert not any("Me llamo Alex" in str(m.content) for m in mensajes_nuevo), \
    "la sesión nueva NO debe contener el historial de Alex"
print("\nOK: el historial completo esta en 'sesion-alex' y 'sesion-nueva' arranca sin contexto.")


=== Historial de 'sesion-alex': 4 mensajes ===
  [human] Me llamo Alex, trabajo con Python y prefiero explicaciones sencillas.
  [ai] ¡Hola Alex! Encantado de conocerte. ¿En qué tema de Python necesitas ayuda? Puedo explicarte conceptos, resolver dudas o guiarte paso a paso. ¡Dime qué te gustaría aprender!
  [human] ¿Cómo puedo leer un archivo CSV?
  [ai] Usa el módulo `csv` o la librería `pandas`.

**Con `csv`:**  
```python
import csv
with open('archivo.csv', newline='') as f:
    lector = csv.reader(f)
    for fila in lector:
        print(fila)
```

**Con `pandas`:**  
```python
import pandas as pd
df = pd.read_csv('archivo.csv')
print(df.head())
```

=== Historial de 'sesion-nueva': 2 mensajes ===
  [human] ¿Cómo puedo leer un archivo CSV?
  [ai] Puedes usar Python:

```python
import pandas as pd
df = pd.read_csv('ruta/archivo.csv')
print(df.head())
```

`read_csv` abre el archivo, lo convierte en una tabla (DataFrame) y `head()` muestra las primeras filas. También puedes usar el

### El mapa del almacén

Vista global: **una fila por hilo** (`thread_id`) y una barra por mensaje guardado. Así se ve el aislamiento sin leer código: cada sesión crece sola y se borra sola.

In [10]:
def mapa_del_almacen(ids):
    print("ALMACÉN DE CHECKPOINTS (InMemorySaver)")
    print("-" * 52)
    for sid in ids:
        n = len(agente.get_state(hilo(sid)).values.get("messages", []))
        print("  hilo '" + sid + "'".ljust(28) + f"{n:2d} mensajes  " + "█" * n)
    print("-" * 52)

mapa_del_almacen(["sesion-alex", "sesion-nueva"])

ALMACÉN DE CHECKPOINTS (InMemorySaver)
----------------------------------------------------
  hilo 'sesion-alex'                            4 mensajes  ████
  hilo 'sesion-nueva'                            2 mensajes  ██
----------------------------------------------------


## Paso 9 · Borrar el historial de una sesión

El borrado es una operación técnica de primera clase (derecho al borrado). `delete_thread` elimina **un solo hilo**: las demás sesiones no se enteran.

In [11]:
checkpointer.delete_thread("sesion-alex")     # borrado selectivo del hilo

tras_borrar = agente.get_state(hilo("sesion-alex")).values.get("messages", [])
otra_sesion = agente.get_state(hilo("sesion-nueva")).values.get("messages", [])

print("'sesion-alex'  tras borrar:", len(tras_borrar), "mensajes")
print("'sesion-nueva' intacta    :", len(otra_sesion), "mensajes")

assert tras_borrar == [], "la sesión Alex debe quedar vacía"
assert len(otra_sesion) == 2, "la otra sesión no debe verse afectada"
print("OK: borrado de una sesión sin tocar las demas.")

'sesion-alex'  tras borrar: 0 mensajes
'sesion-nueva' intacta    : 2 mensajes
OK: borrado de una sesión sin tocar las demas.


## Paso 10 · LlamaIndex en 1 minuto (comparación conceptual)

No implementamos un segundo framework: solo situamos sus nombres para que no suenen ajenos.

| | LangChain (lo visto arriba) | LlamaIndex |
|---|---|---|
| Objeto central | `checkpointer` + `thread_id` por conversación | clase **`Memory`** con `session_id` |
| Corto plazo | estado del hilo persistido por checkpoint (se reinicia al cerrar) | **cola FIFO** de `ChatMessage`; al superar el límite de tokens se vacía (*flush*) |
| Largo plazo | `store` con *namespaces* entre sesiones (`InMemoryStore`, …) | **`Memory Block`**: bloques que conservan/hechan hechos (`StaticMemoryBlock`, `FactExtractionMemoryBlock`, `VectorMemoryBlock`) |
| Vigencia | `create_agent(..., checkpointer=InMemorySaver())` | `ChatMemoryBuffer` está **obsoleto**: usar `Memory` |

```python
from llama_index.core.memory import Memory

memory = Memory.from_defaults(session_id="mi_sesion", token_limit=40000)
# await agent.run(user_msg="...", memory=memory)
```

**Misma idea, distinta caja:** alguien tiene que guardar el historial y reenviarlo al modelo.

Fuentes: [Memory en LlamaIndex](https://developers.llamaindex.ai/python/examples/memory/memory/) · [Memoria de agentes en LlamaIndex](https://developers.llamaindex.ai/python/framework/module_guides/deploying/agents/memory/)

## Conclusiones y preguntas para el grupo

**Lo esencial (3 frases):**
1. El LLM no recuerda nada: la continuidad la aporta **la aplicación**, reenviando el historial.
2. El historial vive en un **hilo** (`thread_id`) con su `checkpointer`: dos hilos = dos sesiones aisladas y borrables por separado.
3. Memoria ≠ conocimiento: guardar más aumenta coste, tokens y riesgo → **minimiza, aísla, borra**.

**Preguntas para el grupo:**
- ¿Por qué el turno 2 de Alex era coherente y el de la sesión nueva no?
- ¿Qué diferencia hay entre el historial de una sesión y la memoria persistente entre sesiones?
- ¿Qué pasaría si dos usuarios compartieran el mismo `thread_id`?
- ¿Qué coste añade reenviar todo el historial en cada turno? ¿Cuándo empezarías a podarlo o resumirlo?
- ¿Qué datos **no** deberían entrar nunca en la memoria de una conversación?

**Siguiente paso:** reto `03_Reto_Memories_Alumno.ipynb` (15 min, sin API).

---
📚 [Memoria en LangChain](https://docs.langchain.com/oss/python/concepts/memory) · [Memoria a corto plazo](https://docs.langchain.com/oss/python/langchain/short-term-memory) · [RunnableWithMessageHistory](https://reference.langchain.com/python/langchain-core/runnables/history/RunnableWithMessageHistory) · [Memory en LlamaIndex](https://developers.llamaindex.ai/python/examples/memory/memory/)